In [10]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [11]:
train_df=pd.read_csv("../data/train_clean.csv")
test_df=pd.read_csv("../data/test_clean.csv")

In [12]:
features=[
    "reynolds",
    "aoa",
    "max_thickness",
    "thickness_location",
    "max_camber",
    "camber_location"
]
X_train=train_df[features]
X_test=test_df[features]
targets=["cm","cl","cd"]

In [9]:
os.makedirs("models/tuned",exist_ok=True)

In [5]:
results={}
for target in targets:
    print(f"Training Linear Regression for {target}")
    
    model= LinearRegression()
    model.fit(X_train, train_df[target])
    
    y_pred=model.predict(X_test)

    mae=mean_absolute_error(test_df[target],y_pred)
    rmse=np.sqrt(mean_squared_error(test_df[target],y_pred))
    r2=r2_score(test_df[target],y_pred)
   
    results[target]={
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }
    joblib.dump(model, f"models/tuned/lr_{target}.pkl")

print("\nLinear Regression completed")
print(results)

Training Linear Regression for cm
Training Linear Regression for cl
Training Linear Regression for cd

Linear Regression completed
{'cm': {'MAE': 0.5624538407116474, 'RMSE': np.float64(0.7502548992077992), 'R2': 0.44234794013300605}, 'cl': {'MAE': 0.1811163547682901, 'RMSE': np.float64(0.23159153746769343), 'R2': 0.9013489758274266}, 'cd': {'MAE': 0.022444931605647447, 'RMSE': np.float64(0.029177048257963564), 'R2': 0.1637881200013639}}


In [ ]:
# Decision Tree

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib

In [7]:
targets = ["cm", "cl", "cd"]

# Hyperparameter grid (safe + standard)
dt_param_grid = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

dt_results = {}

In [8]:
for target in targets:
    print(f"\nTuning Decision Tree for: {target}")

    dt = DecisionTreeRegressor(random_state=42)

    grid_search = GridSearchCV(
        estimator=dt,
        param_grid=dt_param_grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    # Train
    grid_search.fit(X_train, train_df[target])

    # Best model
    best_dt = grid_search.best_estimator_

    # Predictions
    y_pred = best_dt.predict(X_test)

    # Metrics
    mae = mean_absolute_error(test_df[target], y_pred)
    rmse = np.sqrt(mean_squared_error(test_df[target], y_pred))
    r2 = r2_score(test_df[target], y_pred)

    dt_results[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "best_params": grid_search.best_params_
    }

    # Save model
    joblib.dump(best_dt, f"models/tuned/dt_{target}.pkl")

print("\nDecision Tree tuning completed ✔")
print(dt_results)


Tuning Decision Tree for: cm
Fitting 3 folds for each of 36 candidates, totalling 108 fits

Tuning Decision Tree for: cl
Fitting 3 folds for each of 36 candidates, totalling 108 fits

Tuning Decision Tree for: cd
Fitting 3 folds for each of 36 candidates, totalling 108 fits

Decision Tree tuning completed ✔
{'cm': {'MAE': 0.06063584231112519, 'RMSE': np.float64(0.13523503427108924), 'R2': 0.9818814240241209, 'best_params': {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}}, 'cl': {'MAE': 0.026041972796858548, 'RMSE': np.float64(0.04936615560161486), 'R2': 0.9955175563140137, 'best_params': {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}}, 'cd': {'MAE': 0.002434740631135646, 'RMSE': np.float64(0.00581944424083869), 'R2': 0.9667343013838259, 'best_params': {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}}}


In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib

In [11]:
targets = ["cm", "cl", "cd"]

rf_param_grid = {
"n_estimators": [100, 200],
"max_depth": [5, 10, None],
"min_samples_split": [2, 5],
"min_samples_leaf": [1, 2],
"max_features": ["sqrt", 0.8]
}

rf_results = {}

In [13]:
for target in targets:
    print(f"\nTuning Random Forest for: {target}")
    
    rf = RandomForestRegressor(random_state=42)
    
    random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_grid,
    n_iter=5,
    cv=2,
    n_jobs=-1,
    verbose=1,
    random_state=42
    )
    
    # Train
    random_search.fit(X_train, train_df[target])
    
    # Best model
    best_rf = random_search.best_estimator_
    
    # Predictions
    y_pred = best_rf.predict(X_test)
    
    # Metrics
    mae = mean_absolute_error(test_df[target], y_pred)
    rmse = np.sqrt(mean_squared_error(test_df[target], y_pred))
    r2 = r2_score(test_df[target], y_pred)
    
    rf_results[target] = {
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "best_params": random_search.best_params_
    }
    
    # Save model
    joblib.dump(best_rf, f"models/tuned/rf_{target}.pkl")

print("\nRandom Forest tuning completed")
print(rf_results)


Tuning Random Forest for: cm
Fitting 2 folds for each of 5 candidates, totalling 10 fits

Tuning Random Forest for: cl
Fitting 2 folds for each of 5 candidates, totalling 10 fits

Tuning Random Forest for: cd
Fitting 2 folds for each of 5 candidates, totalling 10 fits

Random Forest tuning completed
{'cm': {'MAE': 0.04613422194631638, 'RMSE': np.float64(0.0970050962341369), 'R2': 0.9906774497642892, 'best_params': {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}}, 'cl': {'MAE': 0.019710043107661265, 'RMSE': np.float64(0.03462010077059091), 'R2': 0.9977954858949191, 'best_params': {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}}, 'cd': {'MAE': 0.0018302971370001207, 'RMSE': np.float64(0.004163776884909411), 'R2': 0.982970240073535, 'best_params': {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}}}


In [ ]:
#XG Boost

In [7]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import joblib

In [5]:
targets = ["cm", "cl", "cd"]

xgb_results = {}

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2],
    "reg_lambda": [1, 1.5, 2]
}


In [6]:
for target in targets:
    print(f"\nTuning XGBoost for: {target}")

    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=xgb_param_grid,
        n_iter=10,   # balance between speed and quality
        cv=2,
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    # Train
    random_search.fit(X_train, train_df[target])

    # Best model
    best_xgb = random_search.best_estimator_

    # Predictions
    y_pred = best_xgb.predict(X_test)

    # Metrics
    mae = mean_absolute_error(test_df[target], y_pred)
    rmse = np.sqrt(mean_squared_error(test_df[target], y_pred))
    r2 = r2_score(test_df[target], y_pred)

    xgb_results[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "best_params": random_search.best_params_
    }

    # Save model
    joblib.dump(best_xgb, f"models/tuned/xgb_{target}.pkl")

print("\nXGBoost tuning completed ✔")
print(xgb_results)


Tuning XGBoost for: cm


NameError: name 'XGBRegressor' is not defined

In [ ]:
#Metrics

In [13]:
models = ["lr", "dt", "rf", "xgb"]
all_results = {}
for model_name in models:
    print(f"\nEvaluating {model_name.upper()}...")

    all_results[model_name] = {}

    for target in targets:
        model_path = f"models/tuned/{model_name}_{target}.pkl"
        
        model = joblib.load(model_path)

        y_pred = model.predict(X_test)

        mae = mean_absolute_error(test_df[target], y_pred)
        rmse = np.sqrt(mean_squared_error(test_df[target], y_pred))
        r2 = r2_score(test_df[target], y_pred)

        all_results[model_name][target] = {
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        }

        print(f"{model_name}-{target} → R2: {r2:.4f}")


Evaluating LR...
lr-cm → R2: 0.4423
lr-cl → R2: 0.9013
lr-cd → R2: 0.1638

Evaluating DT...
dt-cm → R2: 0.9819
dt-cl → R2: 0.9955
dt-cd → R2: 0.9667

Evaluating RF...
rf-cm → R2: 0.9907
rf-cl → R2: 0.9978
rf-cd → R2: 0.9830

Evaluating XGB...
xgb-cm → R2: 0.8906
xgb-cl → R2: 0.9796
xgb-cd → R2: 0.8399


In [14]:
rows = []

for model_name, targets_data in all_results.items():
    for target, metrics in targets_data.items():
        rows.append({
            "Model": model_name.upper(),
            "Target": target.upper(),
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2 Score": metrics["R2"]
        })

results_df = pd.DataFrame(rows)

In [15]:
results_df = results_df.sort_values(
    by=["Target", "R2 Score"],
    ascending=[True, False]
)

results_df


,Model,Target,MAE,RMSE,R2 Score
8,RF,CD,0.001830,0.004164,0.982970
5,DT,CD,0.002435,0.005819,0.966734
11,XGB,CD,0.008493,0.012767,0.839904
2,LR,CD,0.022445,0.029177,0.163788
7,RF,CL,0.019710,0.034620,0.997795
4,DT,CL,0.026042,0.049366,0.995518
10,XGB,CL,0.075906,0.105250,0.979625
1,LR,CL,0.181116,0.231592,0.901349
6,RF,CM,0.046134,0.097005,0.990677
3,DT,CM,0.060636,0.135235,0.981881


In [16]:
import os
os.makedirs("results", exist_ok=True)

results_df.to_csv("./results/model_comparison_results.csv", index=False)
print("Saved: model_comparison_results.csv ")

Saved: model_comparison_results.csv 


In [ ]:
#Random Forest achieved the best performance across all aerodynamic coefficients (Cm, Cl, Cd),
# indicating strong capability in capturing nonlinear relationships in airfoil geometry 
# and flow parameters.